### 引言

在学习卷积神经网络时，根据图像的平移不变性和局部性，设计了基于“卷积”的模型结构；而在处理类似于文本这类序列数据时，可以考虑到，序列数据是“前后关联”的，就像一个正在阅读小说的人，他读每一个字时，都会结合之前读过的内容来理解当前的情节。

所以，需要设计一个网络结构，能够拥有“记忆”，这就是循环神经网络的设计思路。

### 结构与原理

#### 1. 循环结构
在数学上，RNN 在当前时间步 $t$ 的隐藏状态 $h_t$ 是由**当前输入 $x_t$** 和 **上一时刻的隐藏状态 $h_{t-1}$** 共同决定的：

$$ h_t = \tanh(W_{xh} \cdot x_t + W_{hh} \cdot h_{t-1} + b) $$

*   $x_t$：当前的输入（比如当前的单词）。
*   $h_{t-1}$：刚才的记忆。
*   $W$ 和 $b$：需要学习的参数。

![循环神经网络](./PPT-figure/循环神经网络.png)


#### 2. 参数共享
RNN 在处理序列的每一步时，使用的是**同一套参数**（权重矩阵 $W$）。这意味着无论序列有多长（是第1个词还是第100个词），网络处理它们的规则是一样的。这大大减少了参数量，也让模型具有了泛化能力。

#### 3. 训练方法：BPTT
RNN 的训练算法叫做**随时间反向传播（Backpropagation Through Time, BPTT）**。
你可以把 RNN 在时间轴上“展开”，把它看作一个很深的神经网络。训练时，误差不仅会向层数反向传播，还会向**过去的时间步**反向传播，以此来更新权重。

### 🌍 常见应用场景

RNN 及其变体（LSTM/GRU）在 Transformer 爆发之前，统治了序列建模领域。

| 任务类型 | 说明 | 典型应用 |
| :--- | :--- | :--- |
| **多对一** | 输入序列 $\rightarrow$ 单个输出 | **情感分析**（输入一句话，输出“正面/负面”） |
| **一对多** | 单个输入 $\rightarrow$ 输出序列 | **图像描述**（输入一张图，输出一段描述文字） |
| **多对多** | 序列 $\rightarrow$ 序列 | **机器翻译**（输入中文句子，输出英文句子） |
| **序列预测** | 预测下一个数据 | **股票预测**、**天气预报**、**文本生成**（打字时的自动联想） |

### 📌 总结

*   **RNN 是什么**：一种专门处理序列数据的神经网络，具有“记忆”功能。
*   **核心机制**：隐藏状态（Hidden State）在时间步之间传递信息。
*   **缺点**：简单的 RNN 难以捕捉长距离依赖（梯度消失）。
*   **进化**：LSTM 和 GRU 通过门控机制解决了长距离记忆问题。
*   **现状**：虽然现在 Transformer（如 BERT, GPT）在很多任务上取代了 RNN，但理解 RNN 是理解这些现代大模型的基础。

### 代码实现

使用莎士比亚的作品作为训练数据，训练一个 RNN 模型，让它能够根据给定的一串字符，生成下一个最可能出现的字符。

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import urllib.request # 用于下载文件

# --- 1. 下载并读取数据 ---
print("正在下载莎士比亚数据集...")
url = 'https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt'
urllib.request.urlretrieve(url, 'shakespeare.txt')

# 读取文本内容
with open('shakespeare.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print(f"文本总长度: {len(text)} 个字符")
print(f"文本前100个字符预览:\n{text[:100]}\n")

正在下载莎士比亚数据集...
文本总长度: 1115394 个字符
文本前100个字符预览:
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You



In [3]:
# --- 2. 数据预处理，创建字符到数字的映射 (词表) ---
# 找出所有不重复的字符
vocab = sorted(set(text))
vocab_size = len(vocab)
print(f"词表大小 (唯一字符数): {vocab_size}")

# 创建两个字典：字符 -> 索引，索引 -> 字符
char2idx = {ch: i for i, ch in enumerate(vocab)}
idx2char = np.array(vocab)

# 将整个文本转换成整数序列
text_as_int = np.array([char2idx[c] for c in text])
print(f"文本前20个字符对应的整数: {text_as_int[:20]}")

词表大小 (唯一字符数): 65
文本前20个字符对应的整数: [18 47 56 57 58  1 15 47 58 47 64 43 52 10  0 14 43 44 53 56]


In [5]:
# --- 3. 设置超参数，构建数据加载器 ---
seq_length = 100  # 每个输入序列的长度
batch_size = 64   # 每次训练用多少个序列

# 创建训练样本
# 我们将文本切分成多个长度为 seq_length+1 的片段
# 输入是前 seq_length 个字符，标签是后 seq_length 个字符（即每个字符的下一个字符）
def create_dataset(data, seq_length):
    inputs = []
    targets = []
    for i in range(0, len(data) - seq_length, seq_length):
        chunk = data[i : i + seq_length + 1]
        inputs.append(chunk[:-1]) # 输入: 从第0个到第99个
        targets.append(chunk[1:]) # 标签: 从第1个到第100个
    return np.array(inputs), np.array(targets)

inputs, targets = create_dataset(text_as_int, seq_length)
print(f"训练样本总数: {len(inputs)}")

# 将数据转换为 PyTorch 的 Tensor 并创建 DataLoader
dataset = torch.utils.data.TensorDataset(
    torch.from_numpy(inputs).long(), 
    torch.from_numpy(targets).long()
)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

训练样本总数: 11153


In [6]:
# --- 4. 定义RNN模型 ---
class SimpleRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super().__init__()
        # 嵌入层：将每个字符的索引映射成一个稠密向量
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        # RNN层：处理序列数据
        self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)
        # 全连接层：将RNN的输出映射回词表大小，用于预测下一个字符
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        # x 的形状: (batch_size, seq_length)
        embedded = self.embedding(x) # 形状: (batch_size, seq_length, embedding_dim)
        output, hidden = self.rnn(embedded) # output形状: (batch_size, seq_length, hidden_dim)
        # 我们只关心每个时间步的输出，所以直接对output进行处理
        return self.fc(output) # 形状: (batch_size, seq_length, vocab_size)

# 实例化模型
embedding_dim = 256
hidden_dim = 512
model = SimpleRNN(vocab_size, embedding_dim, hidden_dim)

# 定义损失函数和优化器
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("模型结构:")
print(model)

模型结构:
SimpleRNN(
  (embedding): Embedding(65, 256)
  (rnn): RNN(256, 512, batch_first=True)
  (fc): Linear(in_features=512, out_features=65, bias=True)
)


In [7]:
# --- 5. 训练模型 ---
num_epochs = 10
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

print(f"在设备 {device} 上开始训练...")

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for batch_idx, (inputs_batch, targets_batch) in enumerate(dataloader):
        inputs_batch = inputs_batch.to(device)
        targets_batch = targets_batch.to(device)
        
        # 前向传播
        outputs = model(inputs_batch)
        # 计算损失
        # 需要将输出和目标调整形状以符合 CrossEntropyLoss 的要求
        loss = criterion(outputs.view(-1, vocab_size), targets_batch.view(-1))
        
        # 反向传播和优化
        optimizer.zero_grad()
        loss.backward()
        ## 梯度裁剪，防止梯度爆炸
        #torch.nn.utils.clip_grad_norm_(model.parameters(), 1)
        optimizer.step()
        
        total_loss += loss.item()
        
        # 每100个批次打印一次进度
        if (batch_idx + 1) % 100 == 0:
            avg_loss = total_loss / 100
            print(f'Epoch [{epoch+1}/{num_epochs}], Step [{batch_idx+1}/{len(dataloader)}], Loss: {avg_loss:.4f}')
            total_loss = 0

print("训练完成！")

在设备 cuda 上开始训练...
Epoch [1/10], Step [100/175], Loss: 2.2817
Epoch [2/10], Step [100/175], Loss: 1.7405
Epoch [3/10], Step [100/175], Loss: 1.6038
Epoch [4/10], Step [100/175], Loss: 1.5281
Epoch [5/10], Step [100/175], Loss: 1.4824
Epoch [6/10], Step [100/175], Loss: 1.4481
Epoch [7/10], Step [100/175], Loss: 1.4204
Epoch [8/10], Step [100/175], Loss: 1.3988
Epoch [9/10], Step [100/175], Loss: 1.3860
Epoch [10/10], Step [100/175], Loss: 1.3711
训练完成！


In [13]:
def generate_text(model, start_string, num_generate=500, temperature=1.0):
    model.eval()
    # 将起始字符串转换为索引
    input_indices = [char2idx[c] for c in start_string]
    input_tensor = torch.tensor([input_indices], dtype=torch.long).to(device)
    
    generated_text = start_string
    
    with torch.no_grad():
        for _ in range(num_generate):
            # 获取模型预测
            predictions = model(input_tensor)
            # 取出最后一个时间步的预测结果
            last_prediction = predictions[0, -1, :] / temperature
            # 使用多项式采样增加随机性
            predicted_id = torch.multinomial(torch.softmax(last_prediction, dim=-1), num_samples=1).item()
            
            # 将预测的字符添加到结果中
            generated_text += idx2char[predicted_id]
            
            # 更新输入，将新预测的字符加入序列
            input_tensor = torch.cat([input_tensor, torch.tensor([[predicted_id]], dtype=torch.long).to(device)], dim=1)
            
    return generated_text

# 生成文本示例
print("\n--- 文本生成示例 ---")
start_string = "ROMEO: "
print(f"起始字符串: {start_string}")
print("生成结果:")
print(generate_text(model, start_string, num_generate=50))


--- 文本生成示例 ---
起始字符串: ROMEO: 
生成结果:
ROMEO: Our man.
Myself;
For I'll am so,
So should not hea
